In [10]:
%pip install pandas
%pip install python-calamine

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd 
import numpy as np
import re
import openpyxl

In [ ]:
data = pd.read_excel(r'C:\Users\USER\Desktop\K4รายงานสต็อกการ์ด พร้อมทุน.xlsx' ,engine='calamine',header=11,usecols="A:F",dtype={'Unnamed: 3': str,'ลด ': str,'คงเหลือ ': str})
data.rename(columns={'Unnamed: 0':'DATE','Unnamed: 1':'Bill','เพิ่ม ':'details','Unnamed: 3':'value','ลด ':'sale','คงเหลือ ':'balance'} ,inplace=True)

# DATE == รหัสสินค้า มีชื่อสินค้าอยู่ ถ้าเป็นค่าว่างให้ดึงชื่อจากแถวก่อนหน้า
data['product_id'] = data.loc[data['DATE'] =='รหัสสินค้า', 'sale']
data['product_id'] = data['product_id'].ffill()

# DATE == คลัง มีชื่อสินค้าอยู่ ถ้าเป็นค่าว่างให้ดึงชื่อจากแถวก่อนหน้า
data['unit'] = data.loc[data['DATE'].astype(str).str.strip() == 'คลัง', 'balance']
data['unit'] = data['unit'].ffill()
data['unit'] = data['unit'].str.extract(r'(\d+)').fillna(0).astype(int)

# แปลงตรงๆ โดยบอกสไตล์ปฏิทินสากลไปก่อน
data['DATE'] = pd.to_datetime(data['DATE'], format='%d/%m/%Y', errors='coerce')

# ลบปีออก 543 ปี (ใช้ DateOffset)
data['DATE'] = data['DATE'] - pd.DateOffset(years=543)
data['DATE'] = pd.to_datetime(data['DATE'])
data.dropna(subset=['DATE'], inplace=True)

แปลงคงเหลือให้เป็นหน่วนเล็กสุด

In [ ]:

def parse_pack_piece(series_val, series_unit):
    """ฟังก์ชันแยกจำนวนแพ็ก (front) และเศษชิ้นย่อย (back) อัตโนมัติ โดยใช้ Logic

    len(str(unit - 1)) กำหนดทศนิยมรายบรรทัด
    """

    def format_by_unit(v, u):
        try:
            val_float = float(v)
            unit_int = int(float(u))

            if unit_int <= 1:
                return f'{val_float:.1f}'

            decimals = len(str(unit_int - 1))
            return f'{val_float:.{decimals}f}'
        except:
            return '0.0'

    # 1. แปลงค่าโดยใช้ .index จาก series_val เดิม เพื่อป้องกัน Index Mismatch
    s_clean = pd.Series(
        [format_by_unit(v, u) for v, u in zip(series_val, series_unit)],
        index=series_val.index,  # <-- ล็อก Index ให้ตรงกับ DataFrame ต้นทาง
    )

    # 2. แยกด้วย .str.split('.') ชัวร์และเร็วกว่า Regex
    split_df = s_clean.str.split('.', expand=True)

    # 3. ดึง front (หน้าจุด)
    front = pd.to_numeric(split_df[0], errors='coerce').fillna(0).astype(int)

    # 4. ดึง back (หลังจุด)
    back = pd.to_numeric(split_df[1], errors='coerce').fillna(0).astype(int)

    return front, back


# ==========================================
# 🚀 โค้ดส่วนการประมวลผลข้อมูล
# ==========================================

# แปลง unit เป็นตัวเลขแท้ๆ ป้องกันคูณแล้วพัง
unit_num = pd.to_numeric(data['unit'], errors='coerce').fillna(1).astype(int)

# 1. แยกหน่วย Import (Value)
data['front_value'], data['back_value'] = parse_pack_piece(
    data['value'], data['unit']
)

# 2. แยกหน่วย Export (Sale)
data['front_sale'], data['back_sale'] = parse_pack_piece(
    data['sale'], data['unit']
)

# 3. แยกหน่วย Balances
data['front_balance'], data['back_balance'] = parse_pack_piece(
    data['balance'], data['unit']
)

# 4. คำนวณข้อย่อยรวม (ใช้ unit_num ที่เป็นตัวเลขแล้ว)
data['import'] = data['back_value'] + (unit_num * data['front_value'])
data['export'] = data['back_sale'] + (unit_num * data['front_sale'])
data['balances'] = data['back_balance'] + (unit_num * data['front_balance'])

## Hybrid Robust Z-score (Global + Rolling) พร้อมคำนวณส่วนต่างและช่วงเวลาบิลหาย

**สิ่งที่เพิ่มเข้ามาใน V3:**
- **คอลัมน์ตรวจสอบบิลย้อนหลัง (Actionable Columns):** สำหรับเคส `🔴 บิลหายทั้งใบ` ระบบจะระบุวันและช่วงวันที่พนักงานบัญชี/คลังสินค้าต้องไปตามหาเอกสารทันที:
  1. `suspect_start_date` (วันที่เริ่มน่าสงสัย) = วันที่ของเข้าล่าสุด + 1 วัน
  2. `suspect_end_date` (วันที่สิ้นสุดการสงสัย) = วันที่ทำรายการเบิกออก/ขายปัจจุบัน
  3. `suspect_date_range` (ช่วงเวลาที่ต้องไปย้อนดูบิล) = รวมเป็นข้อความให้อ่านง่าย เช่น `2026-07-01 ถึง 2026-08-05`

In [17]:
#%pip install scipy
#%pip install openpyxl
#%pip install scikit-learn

In [18]:
import math
import numpy as np
import pandas as pd
from scipy import stats

# ==========================================
# 1. เตรียม Data & Transaction Flags
# ==========================================
df = data[
    [
        'DATE',
        'Bill',
        'details',
        'product_id',
        'import',
        'export',
        'balances',
    ]
].copy()

df.columns = df.columns.str.strip()
df['product_id'] = df['product_id'].astype(str).str.strip()
df['details'] = df['details'].astype(str).str.strip()
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values(by=['product_id', 'DATE']).reset_index(drop=True)

max_date_in_dataset = df['DATE'].max()

# กรองบิลรับคืน / NV
exclude_keywords = ['รับคืน', 'NV']
pattern_exclude = '|'.join(exclude_keywords)
df['is_return_bill'] = df['details'].str.contains(
    pattern_exclude, na=False, case=False
)

# net_export: ถ้าเป็นบิล NV ให้ปรับเป็นค่าลบ เพื่อนำไปหักลบในการคำนวณ Flow จริง
df['net_export'] = np.where(
    df['is_return_bill'], -df['import'], df['export']
)

# ==========================================
# 2. Dynamic Reference Selection (ตัด NV ออกจาก Benchmark)
# ==========================================
# 🎯 ดึงเฉพาะบิลซื้อจริงที่ไม่ใช่ NV มาหา Pack Size และค่ากลาง
import_history = df[(df['import'] > 0) & (~df['is_return_bill'])].copy()


def calculate_adaptive_stats(df_imp):
    if df_imp.empty:
        return pd.DataFrame()

    def get_mode(x):
        m = stats.mode(x, keepdims=False)
        return m.mode if np.isscalar(m.mode) else m.mode[0]

    def get_mode_freq(x):
        m = get_mode(x)
        return np.mean(x == m)

    mode_stats = (
        df_imp.groupby('product_id')['import']
        .agg(
            mode_import_size=get_mode,
            mode_freq=get_mode_freq,
            mad_from_mode=lambda x: np.mean(np.abs(x - get_mode(x))),
        )
        .reset_index()
    )

    med_stats = (
        df_imp.groupby('product_id')['import']
        .agg(
            median_import_size='median',
            mad_val=lambda x: np.median(np.abs(x - np.median(x))),
        )
        .reset_index()
    )

    res = mode_stats.merge(med_stats, on='product_id', how='left')
    res['selected_import_size'] = np.where(
        res['mode_freq'] >= 0.20,
        res['mode_import_size'],
        res['median_import_size'],
    )
    return res


for c in ['mode_import_size', 'median_import_size', 'selected_import_size']:
    if c in df.columns:
        df.drop(columns=[c], inplace=True)

if not import_history.empty:
    stats_df = calculate_adaptive_stats(import_history)
    df = df.merge(
        stats_df[
            [
                'product_id',
                'mode_import_size',
                'median_import_size',
                'selected_import_size',
                'mad_from_mode',
                'mad_val',
            ]
        ],
        on='product_id',
        how='left',
    )

df['selected_import_size'] = df['selected_import_size'].fillna(df['import'])
df['mad_from_mode'] = df['mad_from_mode'].replace(0, 1.0).fillna(1.0)
df['mad_val'] = df['mad_val'].replace(0, 1.0).fillna(1.0)

df['zscore_mode'] = (df['import'] - df['selected_import_size']) / df[
    'mad_from_mode'
]
df['zscore_median'] = 0.6745 * (
    (df['import'] - df['median_import_size']) / df['mad_val']
)

# Outlier Flag (ต้องเป็นบิลซื้อจริง ไม่ใช่ NV)
df['is_outlier_import'] = (
    (df['import'] > 0)
    & (~df['is_return_bill'])
    & (df['selected_import_size'] > 0)
    & (
        (df['import'] >= (df['selected_import_size'] * 2.5))
        | (df['zscore_mode'] >= 2.5)
    )
)

# ==========================================
# 3. Dynamic Idle Days & Movement
# ==========================================
sales_events = df[df['export'] > 0].copy()

if not sales_events.empty:
    sales_events['prev_export_date'] = sales_events.groupby('product_id')[
        'DATE'
    ].shift(1)
    sales_events['inter_sale_days'] = (
        sales_events['DATE'] - sales_events['prev_export_date']
    ).dt.days

    sale_stats = (
        sales_events.groupby('product_id')['inter_sale_days']
        .median()
        .reset_index()
    )
    sale_stats.rename(
        columns={'inter_sale_days': 'median_inter_sale_days'}, inplace=True
    )
    df = df.merge(sale_stats, on='product_id', how='left')
else:
    df['median_inter_sale_days'] = np.nan

df['median_inter_sale_days'] = df['median_inter_sale_days'].fillna(2.0)
df['median_inter_sale_days'] = np.maximum(df['median_inter_sale_days'], 1.0)

df['export_date_temp'] = df['DATE'].where(df['export'] > 0)
df['last_export_date'] = df.groupby('product_id')['export_date_temp'].ffill()

df['days_idle'] = (df['DATE'] - df['last_export_date']).dt.days.fillna(0)
df.drop(columns=['export_date_temp'], inplace=True)

df['current_idle_days_from_max'] = (
    max_date_in_dataset - df['last_export_date']
).dt.days.fillna(0)

df_rev = df.iloc[::-1].set_index('DATE')
df['next_3d_export'] = (
    df_rev.groupby('product_id')['export']
    .rolling('3D', min_periods=1)
    .sum()
    .iloc[::-1]
    .values
    - df['export']
)

df['dynamic_idle_threshold_c1'] = np.ceil(df['median_inter_sale_days'] * 1.5)
df['is_suspected_ghost'] = (
    (df['days_idle'] > df['dynamic_idle_threshold_c1'])
    & (df['balances'] > 0)
    & (df['import'] > 0)
    & (~df['is_return_bill'])
    & (df['next_3d_export'] > 0)
)

df['dynamic_idle_threshold_c2'] = np.ceil(df['median_inter_sale_days'] * 1.8)
df['is_last_record_per_sku'] = df.groupby('product_id')['DATE'].transform(
    'max'
) == df['DATE']

df['is_dead_last_item'] = (
    (df['is_last_record_per_sku'] == True)
    & (df['current_idle_days_from_max'] > df['dynamic_idle_threshold_c2'])
    & (df['balances'] > 0)
    & (df['import'] == 0)
)

# ==========================================
# 4. Reverse Reconciliation (คำนวณย้อนกลับรวม NV)
# ==========================================
df['net_flow'] = df['import'] - df['export']
last_balance_map = df.groupby('product_id')['balances'].last().to_dict()
df['current_last_balance'] = df['product_id'].map(last_balance_map)

# 4.1 Reconciliation สำหรับ Ghost / Outlier
conditions_expected = [
    (df['is_suspected_ghost'] == True) & (df['import'] > 0),
    (df['is_outlier_import'] == True),
]
choices_expected = [
    np.maximum(df['import'] - df['current_last_balance'], 0.0),
    df['selected_import_size'],
]

df['Expected_Import'] = np.select(
    conditions_expected, choices_expected, default=df['import']
)

# 4.2 Reconciliation กรณีติดลบ (Option A & B)
df['is_negative'] = df['balances'] < 0
df['is_first_negative'] = (df['is_negative'] == True) & (
    df.groupby('product_id')['is_negative'].shift(1, fill_value=False) == False
)

sku_max_deficit = (
    df[df['balances'] < 0]
    .groupby('product_id')['balances']
    .min()
    .abs()
    .to_dict()
)
df['max_deficit'] = df['product_id'].map(sku_max_deficit).fillna(0)

# เช็กบิลสั่งซื้อจริงก่อนหน้า (ไม่นับ NV)
sku_has_prior_import = (
    df[(df['import'] > 0) & (~df['is_return_bill'])]['product_id']
    .unique()
    .tolist()
)


def snap_to_pack_size(deficit, pack_size):
    if pack_size <= 0:
        return deficit
    if deficit >= pack_size:
        return math.ceil(deficit / pack_size) * pack_size
    else:
        return pack_size


def reconcile_negative_row(row):
    if not row['is_first_negative']:
        return row['Expected_Import']

    max_def = row['max_deficit']
    pack = row['selected_import_size']
    has_prior = row['product_id'] in sku_has_prior_import

    if has_prior:
        reconciled_val = snap_to_pack_size(max_def, pack)
        return row['Expected_Import'] + reconciled_val
    else:
        return snap_to_pack_size(max_def, pack)


df['Expected_Import'] = df.apply(reconcile_negative_row, axis=1)

# คำนวณ Net Flow และ Balance สะสมย้อนกลับ (เก็บ NV ไว้เดิน Movement จริง)
df['Diff_Import'] = np.where(
    df['import'] > 0, df['import'] - df['Expected_Import'], np.nan
)
df['adjusted_import'] = df['Expected_Import']
df['adjusted_net_flow'] = df['adjusted_import'] - df['net_export']

first_balance_actual = df.groupby('product_id')['balances'].transform('first')
first_net_flow_actual = df.groupby('product_id')['net_flow'].transform('first')
true_initial_balance = first_balance_actual - first_net_flow_actual

df['adjusted_calc_balance'] = (
    true_initial_balance
    + df.groupby('product_id')['adjusted_net_flow'].cumsum()
)

# ==========================================
# 5. สรุปประเภท Anomaly & Estimated Inbound Window
# ==========================================
df['is_stock_out'] = (df['balances'] <= 0) & (df['export'] > 0)
df['is_ghost_stock'] = (
    (df['is_suspected_ghost'] == True)
    & (df['adjusted_calc_balance'] <= 0)
    & (df['balances'] > 0)
)

anomaly_conditions = [
    (df['is_stock_out'] == True),
    (df['is_ghost_stock'] == True),
    (df['is_outlier_import'] == True),
    (df['is_dead_last_item'] == True),
]

anomaly_labels = [
    '🛑 สต็อกหมด/ติดลบ (Stock Out Alert)',
    '🚨 สต็อกผี/บิลเข้าคีย์เกินจนสต็อกบวม (Ghost Stock Alert)',
    '🔵 บิลรับเข้าพุ่งสูงผิดปกติ (Over-Import Anomaly)',
    '📦 สต็อกค้างนานไร้การเคลื่อนไหว (Dead Stock / 1.8x Idle)',
]

df['Anomaly_Type'] = np.select(
    anomaly_conditions, anomaly_labels, default='⚪ ปกติ (Normal)'
)

first_neg_date_map = (
    df[df['is_first_negative'] == True]
    .groupby('product_id')['DATE']
    .min()
    .to_dict()
)
df['first_neg_date'] = df['product_id'].map(first_neg_date_map)


def calculate_inbound_window(row):
    if row['is_first_negative'] or row['is_outlier_import']:
        start_dt = (
            row['first_neg_date']
            if pd.notnull(row['first_neg_date'])
            else row['DATE']
        )
        end_days = int(max(row['median_inter_sale_days'] * 2, 7))
        end_dt = start_dt + pd.Timedelta(days=end_days)
        return (
            f"{start_dt.strftime('%d/%m/%Y')} - {end_dt.strftime('%d/%m/%Y')}"
        )

    return '-'


df['estimated_inbound_window'] = df.apply(calculate_inbound_window, axis=1)
df.drop(columns=['first_neg_date'], inplace=True, errors='ignore')

จับคู่ชื่อ

In [74]:
#%pip install rapidfuzz

In [75]:
search_terms = pd.read_excel(r'C:\Users\KS\Desktop\K4\ชื่อเค4.xlsx' ,engine='calamine',usecols=['ชื่อสินค้า'])
search_terms = search_terms['ชื่อสินค้า'].dropna().tolist()

In [76]:
import re
import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process


# 1. ฟังก์ชัน Clean ข้อความ (Vectorized ผ่าน Pandas)
def clean_series(series):
    return (
        series.fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"[^\w\s]", "", regex=True)
        .str.strip()
    )


# Clean targets
cleaned_targets = clean_series(pd.Series(search_terms)).tolist()
cleaned_targets = [t for t in cleaned_targets if t]

# 2. ดึงเฉพาะ product_id ที่ "ไม่ซ้ำ (Unique)" มา Clean และ Match
unique_products = df["product_id"].drop_duplicates()
cleaned_unique_products = clean_series(unique_products)

# สร้าง Series ลิงก์ระหว่าง product_id เดิม กับ ค่าที่ clean แล้ว
product_clean_map = pd.Series(
    cleaned_unique_products.values, index=unique_products
)

# 3. คำนวณ Similarity Matrix เฉพาะรายการที่ Unique
if cleaned_targets and not cleaned_unique_products.empty:
    # เปลี่ยน scorer เป็น fuzz.QRatio เพื่อ C-Speed เต็มสปีด
    similarity_matrix = process.cdist(
        cleaned_unique_products.tolist(),
        cleaned_targets,
        scorer=fuzz.QRatio,  # ⚡ เร็วกว่า WRatio หลายเท่า
        workers=-1,
    )

    # ดึงค่า Max Similarity ของแต่ละ Unique Product
    max_scores = similarity_matrix.max(axis=1)

    # Map คะแนนกลับไปยัง Unique Product ID
    score_map = pd.Series(max_scores, index=unique_products)

    # 4. Map คะแนนกลับเข้า DataFrame หลัก
    df["max_similarity"] = df["product_id"].map(score_map).fillna(0.0)
else:
    df["max_similarity"] = 0.0

# 5. กรองเฉพาะรายการที่คะแนนถึงเกณฑ์
threshold = 90
filtered_df = df[df["max_similarity"] >= threshold].copy()

In [19]:
export_cols = [
    'DATE',
    'Bill',
    'details',
    'product_id',
    'import',
    'export',
    'balances',
    'mode_import_size',
    'median_import_size',
    'Expected_Import',
    'adjusted_calc_balance',
    'estimated_inbound_window',
    'is_outlier_import',
    'Anomaly_Type',
]
df[export_cols].to_excel(
    r'C:\Users\KS\Desktop\product_out1.xlsx',index=False,engine='openpyxl')